In [11]:
using Gridap
using GridapMakie, CairoMakie, FileIO
using Gridap.FESpaces
using Gridap.ReferenceFEs
using Gridap.Arrays
using Gridap.Algebra
using Gridap.Geometry
using Gridap.Fields
using Gridap.CellData
using FillArrays
using Test
using InteractiveUtils

function GradientDescent(;solveSE, solveAE, spaces, dΩ, dΓ=nothing, Q, J, ∇f, iter_max=1000, tol=1e-3, P=x->x, u0=nothing, w=nothing, s_min=nothing, sminargs=nothing, armijoparas=(ρ=1/2, α_0=1, α_min=1/2^5, σ=1e-4), Δt=0.05, t0=0.0, tF, saveall::Bool=false)

	Trialspace, Testspace, Qspace = spaces                                  # Extract spaces
	
	q = [(t,interpolate_everywhere(Q(t),Qspace(t))) for t=t0:Δt:tF]    # Initialize u with some random values and apply projection
	qfun(t)=find(q,t)

	y, cacheSE, A_SE = solveSE(Q,Trialspace,Testspace,w=w,dΩ=dΩ,dΓ=dΓ)  # initial SE solve
	#y = FEFunction(Trialspace, y_dof)
	yfun(t)=find(y,t)
	println("y computed")

	p, cacheAE, A_AE = solveAE(yfun,q,Trialspace,Testspace,dΩ=dΩ,dΓ=dΓ)      # initial AE solve
	#p = FEFunction(Testspace, p_dof)
	println("p computed")
	
	cost = J(y, q)
	fgrad =  ∇f(q, p, y)														# Compute initial gradient
	L2fgrad_save = L2norm(fgrad)                                       # Compute norm of initial gradient

	if saveall
		qs=[q]                     # save the solutions - only if really necessary
		ys=[y]
		ps=[p]
		costs=[cost]
	else
		qs,ys,ps=[],[],[]
	end
	for k=1:iter_max
		println("entered for loop, E=$cost")
		
		q_new = y_new = cost_new = qfunnew = nothing
		s = s_min(q,y,p,fgrad;solveSE=solveSE, solveAE=solveAE, spaces=spaces, w=w, dΩ=dΩ, dΓ=dΓ, Δt=Δt, t0=t0, tF=tF)
		println("s_min = $s")
		# q_new = [(t,interpolate_everywhere((qfun(t) - s*grad)*q_pos, Qspace(t))) for (t,grad) in fgrad] |> Proj #interpolate_everywhere(q - s*fgrad,Qspace) |> P			# in most cases interpolate instead of interpolate_everywhere works as well
		q_new = [(t,interpolate_everywhere((qfun(t) + s*grad), Qspace(t))) for (t,grad) in fgrad] #interpolate_everywhere(q - s*fgrad,Qspace) |> P			# in most cases interpolate instead of interpolate_everywhere works as well
		
		if !isdir("tmp")
			mkdir("tmp")
		  end
		  createpvd("results") do pvd
			# pvd[0] = createvtk(Ω, "tmp/results_0" * ".vtu", cellfields=["u" => uh0])
			for (tn, uhn) in fgrad
			  pvd[tn] = createvtk(Ω, "tmp/results_$tn" * ".vtu", cellfields=["u" => uhn])
			end
		  end
		
		qfunnew=t->find(q_new,t)
		y_new, cacheSE = solveSE(qfunnew,Trialspace,Testspace;w=w,dΩ=dΩ,dΓ=dΓ)
		#y_new = FEFunction(Trialspace, y_dof)

		cost_new = J(y_new,q_new)
		q = q_new
		y = y_new
		qfun = qfunnew
		cost = cost_new
		
		if saveall
			push!(qs,q)
			push!(ys,y)
			push!(costs,cost)
		end
		yfun=t->find(y,t)
		p, cacheAE = solveAE(yfun,q,Trialspace,Testspace;dΩ=dΩ,dΓ=dΓ)
		#p = FEFunction(Testspace, p_dof)

		fgrad = ∇f(q, p, y)
		L2fgrad = L2norm(fgrad)
		push!(ps,p)

		if L2fgrad < tol*L2fgrad_save                           # loop break condition - better ideas are appreciated
			break
		end
	end
	return saveall ? (ys,qs,ps,costs) : (y,q,p,cost)                    	# give back either all saved variables or only end result
end


domain = (-1,+1,-1,+1)
partition = (20,20)
model = CartesianDiscreteModel(domain,partition) |> simplexify
order = 1
reffe = ReferenceFE(lagrangian,Float64,order)
Testspace = TestFESpace(model,reffe,conformity=:H1) ###### conformity correct?
Trialspace = TransientTrialFESpace(Testspace)                                # maybe add a function for/if Dirichlet conditions

Uspace = FESpace(model, reffe, conformity=:H1)

degree = 2*order                                                    # degree of the method used for approximating integrals over Ω
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)                                              # make the measure dΩ
Γ = BoundaryTriangulation(model)                                    # triangulate the boundary ∂Ω
dΓ = Measure(Γ,degree)                                              # measure on Γ

χ(x,a,b;g::Function=x->x) = a≤x≤b ? g(x) : 0# all(a .≤ [xx for xx in x] .≤ b) ? g(x) : 0.0
q_pos(x) = χ(x[1], -0.9, -0.7) * χ(x[2], 0, 0.20) + χ(x[1], 0.2, 0.4) * χ(x[2], -0.5, 0)#χ(x, [-0.9, 0], [-0.7, 0.2])  + χ(x, [0.2, -0.5], [0.4, 0])
ρ(x)=1.0#1.225
c(x)=1.0#1020.0
k(x)=1.0#15.0
h(x)=1.0#0.7
Toutdoor(x,t)=1.0
Tout(t)=x->Toutdoor(x,t)
Q(x,t)=χ(t,0.0,10.0)*1000.0*q_pos(x)*0.0
Qt(t)=x->Q(x,t)
price(t)=0.0
Tini(x)=20.0
t0=0.0
tF=10.0
TIni=interpolate_everywhere(Tini, Uspace(t0))
Tfin=interpolate_everywhere(100.0, Uspace(tF))
Δt = 0.05

Proj(a,b,z) = min(max(a,z),b)
# Proj(z) = map(x->Proj(a,b,x),z)
a=0.0
b=1000.0
Proj(z) = [(t,FEFunction(Uspace,map(x->Proj(a,b,x), get_free_dof_values(zz)))) for (t,zz) in z]
# [(t,interpolate_everywhere(map(x->P(a,b,x),get_free_dof_values(zz)),Uspace(t))) for (t,zz) in z]

γ = 1.0
function E(T,Q)
	E=0.0
	for (t,QQ) in Q
		E+=Δt*price(t)^2*γ*∑(∫(QQ*QQ)*dΩ)
	end
	tmp=last(T)[2]-Tfin
	println(E/2.0)
	E+=∑(∫(tmp*tmp)*dΩ)
	return E/2.0
end

∇e(Q::Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}},
T::Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}},
W::Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}}) = [(Q[k][1],2*price(T[k][1])^2*Q[k][2]-ρ*W[k][2]*c) for k=1:length(Q)]                                                # gradient of reduced cost
function ∇e(Qt::Function,
T::Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}},
W::Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}})
	println(T)
	println(W)
	println("AAA")
	return collect((T[k][1],interpolate_everywhere(2*price(T[k][1])^2*Qt(T[k][1])-ρ*W[k][2]*c,Uspace(T[k][1]))) for k=1:length(W))
end

function find(y,s)
	tsave,ysave=t0,nothing
	for (t,yy) in y
		ysave=yy
		break
	end
	for (t,yy) in y
		if t≈s
			return yy
		elseif t ≥ s
			return interpolate_everywhere(((s-tsave)*ysave+(t-s)*yy)/Δt, Uspace(s))
		end
		tsave,ysave = t,yy
	end
end

L2norm(u)=√((tF-t0)*∑(Δt*∑(∫(uu⋅uu)*dΩ) for (t,uu) in u))
L2skp(u)=(tF-t0)*∑(Δt*∑(∫(uu⋅uu)*dΩ) for (t,uu) in u)
ls = LUSolver()
θ = 0.5
solver = ThetaMethod(ls, Δt, θ)


function SEsolver(Qt,Trialspace,Testspace;w=nothing,dΩ,dΓ=nothing,cache=nothing,A=nothing,y_dof=fill(0.0, num_free_dofs(Testspace)))
	a_SE_tconst(t, dtT, ϕ) = ∫(c*dtT*ϕ*ρ)dΩ
	a_SE_tnonconst(t, T, ϕ) = ∫(k * ∇(T) ⋅ ∇(ϕ))dΩ + ∫(h*T*ϕ)dΓ
	l_SE(t, ϕ) = ∫(Qt(t) * ϕ)dΩ + ∫(Tout(t) * ϕ * h)dΓ	
	op_SE = TransientLinearFEOperator((a_SE_tnonconst, a_SE_tconst), l_SE, Trialspace, Testspace, constant_forms=(true, true))
	T = solve(solver, op_SE, t0, tF, TIni)

	return [(t0, TIni), collect((t, FEFunction(Trialspace, copy(get_free_dof_values(TT)))) for (t, TT) in T)...], 0.0, 0.0
end

function AEsolver(T,Q,Trialspace,Testspace;dΩ,dΓ=nothing,cache=nothing,A=nothing,W_dof=fill(0.0, num_free_dofs(Testspace)))
	a_AE_tconst(t, dtW, ψ) = ∫(c*dtW*ψ*ρ)dΩ
	a_AE_tnonconst(t, W, ψ) = ∫(k * (∇(W) ⋅ ∇(ψ)))dΩ - ∫(h*W*ψ)dΓ
	l_AE(t, ψ) = ∫(0.0*ψ)dΩ
	op_AE = TransientLinearFEOperator((a_AE_tnonconst, a_AE_tconst), l_AE, Trialspace, Testspace, constant_forms=(true, true))
	W_end=interpolate_everywhere(-γ*(T(tF)-Tfin)/c/ρ, Uspace(tF))
	W = solve(ThetaMethod(LUSolver(), Δt, θ), op_AE, t0, tF, W_end)

	W_copy = [collect((t, FEFunction(Trialspace, copy(get_free_dof_values(WW)))) for (t, WW) in W)...]
	W_copy_reversed = reverse(W_copy)
	push!(W_copy_reversed,(tF,W_end))
	return W_copy_reversed, 0.0, 0.0
end

function s_min(Q,T,W,gradient;solveSE, solveAE, spaces, w=nothing, dΩ, dΓ=nothing, s_ini=nothing, Δt=0.05, t0=0.0, tF)
	Trialspace, Testspace, FEspace = spaces
	gradfun(t)=find(gradient,t)
	Sv,_ = solveSE(gradfun,Trialspace,Testspace;w=w,dΩ=dΩ,dΓ=dΓ)
	L2NormSquaredOfv = (tF-t0)*∑(Δt*∑(∫(uu⋅uu)*dΩ) for (t,uu) in gradient)                                 # ||v||^2
	L2NormSquaredSv = (tF-t0)*∑(Δt*∑(∫(uu⋅uu)*dΩ) for (t,uu) in Sv)
	L2NormSquaredpv = (tF-t0)*∑(Δt*∑(∫(uu⋅uu*price*price)*dΩ) for (t,uu) in gradient)

	return -L2NormSquaredOfv/(2*(L2NormSquaredpv+γ/2*L2NormSquaredSv)) # -s_min because originally v=-∇e but calculating this costs much more time
end

(ys,qs,ps,costs) = GradientDescent(;solveSE=SEsolver, 
solveAE=AEsolver, 
spaces=(Trialspace, Testspace, Uspace), 
dΩ=dΩ, 
dΓ=dΓ, 
Q=Qt, 
J=E, 
∇f=∇e, 
P=Proj, 
s_min=s_min,
sminargs=nothing, 
saveall=true, 
tol=1e-5, 
iter_max=6,
armijoparas=(ρ=1/2, α_0=100, α_min=1/2^20, σ=1e-4), 
Δt=Δt, 
t0=t0, 
tF=tF)

y computed
p computed
0.0
entered for loop, E=19601.997320126015
s_min = -1.903713781710509
0.0
entered for loop, E=19108.798051525653
s_min = -4.169603317953853
0.0
entered for loop, E=16739.855609389026
s_min = -3.2369355366587027
0.0
entered for loop, E=10668.777765966122
s_min = -2.646391546207801
0.0
entered for loop, E=1495.9664777008297
s_min = -2.4677584905764824
0.0
entered for loop, E=23363.245687252656
s_min = -2.39620066931733
0.0
entered for loop, E=397478.99481124594
s_min = -2.3621405049969875
0.0
entered for loop, E=3.5990386561151394e6
s_min = -2.3444134588377095
0.0
entered for loop, E=2.7912357811758857e7
s_min = -2.3356797647616694
0.0
entered for loop, E=2.088369384688262e8
s_min = -2.333319777269274
0.0
entered for loop, E=1.5697369434539611e9
s_min = -2.3365526558686547
0.0
entered for loop, E=1.2017557879434557e10
s_min = -2.3454115390989436
0.0
entered for loop, E=9.393047765644191e10
s_min = -2.360325851876603
0.0
entered for loop, E=7.475312074998105e11
s_min

(Vector{Tuple{Float64, SingleFieldFEFunction{GenericCellField{ReferenceDomain}}}}[[(0.0, SingleFieldFEFunction()), (0.05, SingleFieldFEFunction()), (0.1, SingleFieldFEFunction()), (0.15000000000000002, SingleFieldFEFunction()), (0.2, SingleFieldFEFunction()), (0.25, SingleFieldFEFunction()), (0.3, SingleFieldFEFunction()), (0.35, SingleFieldFEFunction()), (0.39999999999999997, SingleFieldFEFunction()), (0.44999999999999996, SingleFieldFEFunction())  …  (9.55, SingleFieldFEFunction()), (9.600000000000001, SingleFieldFEFunction()), (9.650000000000002, SingleFieldFEFunction()), (9.700000000000003, SingleFieldFEFunction()), (9.750000000000004, SingleFieldFEFunction()), (9.800000000000004, SingleFieldFEFunction()), (9.850000000000005, SingleFieldFEFunction()), (9.900000000000006, SingleFieldFEFunction()), (9.950000000000006, SingleFieldFEFunction()), (10.000000000000007, SingleFieldFEFunction())], [(0.0, SingleFieldFEFunction()), (0.05, SingleFieldFEFunction()), (0.1, SingleFieldFEFunction(

In [15]:
test = ys[8]
if !isdir("tmp")
    mkdir("tmp")
  end
  
  createpvd("results") do pvd
    # pvd[0] = createvtk(Ω, "tmp/results_0" * ".vtu", cellfields=["u" => uh0])
    for (tn, uhn) in test
      pvd[tn] = createvtk(Ω, "tmp/results_$tn" * ".vtu", cellfields=["u" => uhn])
    end
  end

202-element Vector{String}:
 "results.pvd"
 "tmp/results_0.0.vtu"
 "tmp/results_0.05.vtu"
 "tmp/results_0.1.vtu"
 "tmp/results_0.15000000000000002.vtu"
 "tmp/results_0.2.vtu"
 "tmp/results_0.25.vtu"
 "tmp/results_0.3.vtu"
 "tmp/results_0.35.vtu"
 "tmp/results_0.39999999999999997.vtu"
 ⋮
 "tmp/results_9.600000000000001.vtu"
 "tmp/results_9.650000000000002.vtu"
 "tmp/results_9.700000000000003.vtu"
 "tmp/results_9.750000000000004.vtu"
 "tmp/results_9.800000000000004.vtu"
 "tmp/results_9.850000000000005.vtu"
 "tmp/results_9.900000000000006.vtu"
 "tmp/results_9.950000000000006.vtu"
 "tmp/results_10.000000000000007.vtu"

In [4]:
function savePVDall(ys, qs, ps)
    for k = 1:length(ys)
        if !isdir("tmpse$k")
            mkdir("tmpse$k")
        end
        createpvd("results_se$k") do pvd
            for (tn, uhn) in ys[k]
                pvd[tn] = createvtk(Ω, "tmpse$k/results_se{$k}_$tn" * ".vtu", cellfields=["u" => uhn])
            end
        end
    end
    for k = 1:length(ps)
        if !isdir("tmpadj$k")
            mkdir("tmpadj$k")
        end
        createpvd("results_adj$k") do pvd
            for (tn, uhn) in ps[k]
                pvd[tn] = createvtk(Ω, "tmpadj$k/results_adj{$k}_$tn" * ".vtu", cellfields=["u" => uhn])
            end
        end
    end
    for k = 1:length(qs)
        if !isdir("tmpcont$k")
            mkdir("tmpcont$k")
        end
        createpvd("results_cont$k") do pvd
            for (tn, uhn) in qs[k]
                pvd[tn] = createvtk(Ω, "tmpcont$k/results_cont{$k}_$tn" * ".vtu", cellfields=["u" => uhn])
            end
        end
    end
end

ysave = ys
qsave = qs
psave = ps
costsave = costs

savePVDall(ys, qs, ps)

In [44]:
get_node_coordinates(Ω)

441-element Vector{VectorValue{2, Float64}}:
                 VectorValue{2, Float64}(-1.0, -1.0)
                 VectorValue{2, Float64}(-0.9, -1.0)
                 VectorValue{2, Float64}(-0.8, -1.0)
                 VectorValue{2, Float64}(-0.7, -1.0)
                 VectorValue{2, Float64}(-0.6, -1.0)
                 VectorValue{2, Float64}(-0.5, -1.0)
  VectorValue{2, Float64}(-0.3999999999999999, -1.0)
 VectorValue{2, Float64}(-0.29999999999999993, -1.0)
 VectorValue{2, Float64}(-0.19999999999999996, -1.0)
 VectorValue{2, Float64}(-0.09999999999999998, -1.0)
                            ⋮
   VectorValue{2, Float64}(0.20000000000000018, 1.0)
   VectorValue{2, Float64}(0.30000000000000004, 1.0)
   VectorValue{2, Float64}(0.40000000000000013, 1.0)
                   VectorValue{2, Float64}(0.5, 1.0)
    VectorValue{2, Float64}(0.6000000000000001, 1.0)
    VectorValue{2, Float64}(0.7000000000000002, 1.0)
                   VectorValue{2, Float64}(0.8, 1.0)
    VectorValue{2, Float